# NB2 - Depth Pro (metric depth)
Input : ROOT/images/*.jpg|png
Output: ROOT/out/depth/<stem>.npz  (depth in metres, f_px)  +  ROOT/out/depth_vis/<stem>.png
Runtime: GPU (T4)

In [ ]:
!pip -q install git+https://github.com/apple/ml-depth-pro.git matplotlib
!mkdir -p /content/checkpoints
!wget -q -nc https://ml-site.cdn-apple.com/models/depth-pro/depth_pro.pt -P /content/checkpoints



In [ ]:
!pip install --upgrade numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob, json
import numpy as np, torch, cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import depth_pro

os.chdir('/content')   # the model looks for ./checkpoints/depth_pro.pt

# ---- CHOOSE WHERE TO WORK (use the same two lines in the GeoCalib notebook) ----
TARGET    = 'demo'   # 'accuracy' = tape-measured photos | 'demo' = images for the final pipeline
OVERWRITE = False        # True = recompute even if an output already exists
# --------------------------------------------------------------------------------
BASE    = f'/content/drive/MyDrive/cv5570_risk/{TARGET}'
IMG_DIR = f'{BASE}/images'
OUT_D, OUT_V = f'{BASE}/depth', f'{BASE}/depth_vis'
for d in (IMG_DIR, OUT_D, OUT_V): os.makedirs(d, exist_ok=True)

assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
model, transform = depth_pro.create_model_and_transforms(device=torch.device('cuda'), precision=torch.half)
model.eval()
print('images folder:', IMG_DIR)

In [ ]:
paths = sorted(p for p in glob.glob(f'{IMG_DIR}/*') if p.lower().endswith(('.jpg', '.jpeg', '.png')))
print(len(paths), 'images in', IMG_DIR)
assert paths, 'No images found - put them in the images folder first (see the upload cell above)'

for p in paths:
    stem = os.path.splitext(os.path.basename(p))[0]
    npz_path = f'{OUT_D}/{stem}.npz'
    im = cv2.imread(p)
    if im is None:
        print(f'{stem}: cannot read this file (HEIC / corrupt?) - skipped'); continue
    H, W = im.shape[:2]

    if os.path.exists(npz_path) and not OVERWRITE:
        if np.load(npz_path)['depth'].shape == (H, W):
            print(f'{stem}: already done (skipped)'); continue
        print(f'{stem}: saved map has a different size than the image -> recomputing')

    image, _, f_px = depth_pro.load_rgb(p)             # f_px is None if the image has no EXIF
    with torch.no_grad():
        pred = model.infer(transform(image), f_px=f_px)
    depth = pred['depth'].squeeze().float().cpu().numpy()          # metres, shape (H, W)
    f_est = float(pred['focallength_px'])
    if depth.shape != (H, W):
        print(f'WARNING {stem}: depth map {depth.shape[::-1]} vs image {(W, H)} - rotated photo? later notebooks will flag it')
    np.savez_compressed(npz_path, depth=np.clip(depth, 0, 60000).astype(np.float16), f_px=f_est)

    plt.figure(figsize=(6, 4)); plt.imshow(depth, cmap='turbo', vmin=0, vmax=np.percentile(depth, 98))
    plt.colorbar(label='metres'); plt.axis('off'); plt.title(stem)
    plt.savefig(f'{OUT_V}/{stem}.png', dpi=100, bbox_inches='tight'); plt.close()
    print(f'{stem}: saved | f_px={f_est:.0f} | median depth={np.median(depth):.1f} m')

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from PIL import Image
import os

# Find processed images and display original side-by-side with depth map
for p in paths:
    stem = os.path.splitext(os.path.basename(p))[0]
    vis_path = f"{OUT_V}/{stem}.png"

    if os.path.exists(vis_path):
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))

        # Original Image
        orig_img = Image.open(p)
        axes[0].imshow(orig_img)
        axes[0].set_title(f"Original: {stem}")
        axes[0].axis("off")

        # Depth Visualization
        depth_vis = Image.open(vis_path)
        axes[1].imshow(depth_vis)
        axes[1].set_title(f"Depth Visual: {stem}")
        axes[1].axis("off")

        plt.tight_layout()
        plt.show()

In [ ]:
import shutil
from google.colab import files

# Zip all generated output maps and download them
output_zip_path = "/content/depth_pro_outputs"
shutil.make_archive(output_zip_path, 'zip', BASE)

print("Downloading your depth maps and visualizations...")
files.download(f"{output_zip_path}.zip")

Depth Pro maps are only used for the struck-by cross-check (NB4, DIST_METHOD='depthpro').
Sanity check: open a few PNGs in out/depth_vis - nearby things should be blue, far things red.